In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine,text 

In [2]:
database = 'DATA_SCIENCE'
servername = r'SerhanOztasAsus\SRNINTLSQLSERVER'
engine = create_engine(f"mssql+pyodbc://{servername}/{database}"
                       "?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes")
print(engine)

Engine(mssql+pyodbc://SerhanOztasAsus\SRNINTLSQLSERVER/DATA_SCIENCE?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes)


In [31]:
with engine.connect() as conn:
    df_messy_sales_orders = pd.read_sql(text('SELECT * FROM [DBO].[messy_sales_orders]'),conn)
    df_customer_reference = pd.read_sql(text('SELECT * FROM [dbo].[customer_reference]'),conn)
    df_product_reference  = pd.read_sql(text('SELECT * FROM [dbo].[product_reference]'),conn)
    df_region_targets     = pd.read_sql(text('SELECT * FROM [dbo].[region_targets]'),conn)



df_messy_sales_orders.reset_index()


,index,row_id,order_id,customer_id,customer_email,region,country,product_sku,product_name,sales_channel,...,discount_pct,tax_rate,shipping_cost,payment_method,coupon_code,fulfillment_status,notes,order_id_key,customer_id_key,product_sku_key
0,0,1,ORD-000001,CUST-00002,customer1@example.com,Europe,Germany,SKU-002,Bluetooth Speaker,Offline,...,0.01,0.060,5.74,CARD,,delivered,,ORD-000001,CUST-00002,SKU-002
1,1,2,ORD-000002,CUST-00003,customer2@example.com,Asia Pacific,Japan,SKU-003,Standing Desk,Online,...,0.02,0.070,6.49,PayPal,,delivered,,ORD-000002,CUST-00003,SKU-003
2,2,3,ORD-000003,CUST-00004,customer3@example.com,Latin America,Brazil,SKU-004,Coffee Grinder,Offline,...,0.03,0.080,7.24,wire,,delivered,,ORD-000003,CUST-00004,SKU-004
3,3,4,ORD-000004,CUST-00005,customer4@example.com,Middle East,UAE,SKU-005,Skin Care Kit,Online,...,0.04,0.090,7.99,Cash,,delivered,,ORD-000004,CUST-00005,SKU-005
4,4,5,ORD-000005,CUST-00006,customer5@example.com,Africa,South Africa,SKU-006,Yoga Mat,Offline,...,0.05,0.100,8.74,,NaN,delivered,,ORD-000005,CUST-00006,SKU-006
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11995,11995,11996,ORD-011996,CUST-00497,customer11996@example.com,Asia Pacific,Spain,SKU-005,Skin Care Kit,Online,...,0.26,0.130,24.49,Credit Card,,delivered,,ORD-011996,CUST-00497,SKU-005
11996,11996,11997,ORD-011997,CUST-00498,customer11997@example.com,latam,Kenya,SKU-006,Yoga Mat,Offline,...,0.27,0.050,25.24,Credit Card,,delivered,,ORD-011997,CUST-00498,SKU-006
11997,11997,11998,ORD-011998,CUST-00499,customer11998@example.com,Middle East,Australia,SKU-007,Camping Lamp,Online,...,0.28,0.060,25.99,Credit Card,N/A,delivered,,ORD-011998,CUST-00499,SKU-007
11998,11998,11999,ORD-011999,CUST-00500,customer11999@example.com,Africa,United States,SKU-008,SQL Handbook,Offline,...,0.29,0.070,26.74,credit card,BULK25,delivered,,ORD-011999,CUST-00500,SKU-008


In [32]:
df_messy_sales_orders.columns

Index(['row_id', 'order_id', 'customer_id', 'customer_email', 'region',
       'country', 'product_sku', 'product_name', 'sales_channel',
       'order_priority', 'order_date', 'ship_date', 'units_sold', 'unit_price',
       'unit_cost', 'discount_pct', 'tax_rate', 'shipping_cost',
       'payment_method', 'coupon_code', 'fulfillment_status', 'notes',
       'order_id_key', 'customer_id_key', 'product_sku_key'],
      dtype='str')

In [33]:
# N/A, NA, null, None,"",""Lesson11-Refresh Connection.ipynb"

missing_words = ['', " ", "N/A", "NA", "null", "None", "n/a", "na", "Null", "NULL", "none", "NoneType", "not available"]

df_messy_sales_orders.columns

for column in df_messy_sales_orders.columns:
    #print(df_messy_sales_orders[column].dtype)
    df_messy_sales_orders[column] = df_messy_sales_orders[column].astype('string').str.strip()  # ltrim, rtrim and etc
    df_messy_sales_orders[column] = df_messy_sales_orders[column].replace(missing_words,pd.NA) 


#print(df_messy_sales_orders['coupon_code'] == ' welcome10 ')
df_messy_sales_orders['coupon_code'].str.contains('welcome10', case=False, na=False) 
# select distinct coupon_code,count(*) from messy_sales_orders where coupon_code like '%welcome10%' group by coupon_code

df_messy_sales_orders['coupon_code'].drop_duplicates() # 4 columns, 
df_messy_sales_orders['coupon_code'].unique()

# df_messy_sales_orders.loc[df_messy_sales_orders['coupon_code'].str.contains('welcome10', case=False, na=False), :].groupby('coupon_code').size().reset_index(name='count')
# select distinct coupon_code,count(*) from messy_sales_orders where coupon_code like '%welcome10%' group by coupon_code
df_messy_sales_orders.loc[df_messy_sales_orders['coupon_code'].isna() | df_messy_sales_orders['coupon_code']
                          .notnull() , :].groupby('coupon_code',dropna=False).size().reset_index(name='count')

# "" -> N/A


,coupon_code,count
0,BULK25,576
1,FREESHIP,406
2,welcome10,748
3,<NA>,10270


In [34]:


df_messy_sales_orders.groupby("coupon_code", dropna=False) \
    .size() \
    .fillna({"coupon_code": ""}).reset_index()




,coupon_code,0
0,BULK25,576
1,FREESHIP,406
2,welcome10,748
3,<NA>,10270


In [7]:
s = '  hello world    '
l = s.split(" ")
print(l)
print(s)
s = s.strip()
print(s)
l = s.split(" ")
print(l)

['', '', 'hello', 'world', '', '', '', '']
  hello world    
hello world
['hello', 'world']


In [35]:
df_messy_sales_orders['discount_pct'] = df_messy_sales_orders['discount_pct']\
    .astype('str').replace('75%', '0.75')



# df_messy_sales_orders.loc[
#     df_messy_sales_orders["discount_pct"] == "75%",
#     "discount_pct"
# ] = "0.75"

df_messy_sales_orders.loc[
    df_messy_sales_orders["discount_pct"]== "0.75"]

,row_id,order_id,customer_id,customer_email,region,country,product_sku,product_name,sales_channel,order_priority,...,discount_pct,tax_rate,shipping_cost,payment_method,coupon_code,fulfillment_status,notes,order_id_key,customer_id_key,product_sku_key
108,109,ORD-000109,CUST-00110,customer109@example.com,Europe,South Africa,SKU-006,Yoga Mat,Offline,M,...,0.75,0.060,7.99,<NA>,<NA>,delivered,<NA>,ORD-000109,CUST-00110,SKU-006
217,218,ORD-000218,CUST-00219,customer218@example.com,Asia Pacific,Spain,SKU-003,Standing Desk,Online,L,...,0.75,0.070,10.99,Credit Card,<NA>,delivered,<NA>,ORD-000218,CUST-00219,SKU-003
326,327,ORD-000327,CUST-00328,customer327@example.com,Latin America,Japan,SKU-008,SQL Handbook,Offline,C,...,0.75,0.080,13.99,PayPal,<NA>,delivered,<NA>,ORD-000327,CUST-00328,SKU-008
435,436,ORD-000436,CUST-00437,customer436@example.com,Middle East,Canada,SKU-005,Skin Care Kit,Online,H,...,0.75,0.090,16.99,Credit Card,<NA>,delivered,<NA>,ORD-000436,CUST-00437,SKU-005
544,545,ORD-000545,CUST-00046,customer545@example.com,Africa,Australia,SKU-002,Bluetooth Speaker,Offline,M,...,0.75,0.100,19.99,Credit Card,<NA>,delivered,<NA>,ORD-000545,CUST-00046,SKU-002
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11444,11445,ORD-011445,CUST-00446,customer11445@example.com,Latin America,South Africa,SKU-006,Yoga Mat,Offline,M,...,0.75,0.110,4.99,<NA>,<NA>,delivered,<NA>,ORD-011445,CUST-00446,SKU-006
11553,11554,ORD-011554,CUST-00055,customer11554@example.com,Middle East,Spain,SKU-003,Standing Desk,Online,L,...,0.75,0.120,7.99,Credit Card,<NA>,delivered,<NA>,ORD-011554,CUST-00055,SKU-003
11771,11772,ORD-011772,CUST-00273,customer11772@example.com,North America,Canada,SKU-005,Skin Care Kit,Online,H,...,0.75,0.050,13.99,Credit Card,<NA>,delivered,<NA>,ORD-011772,CUST-00273,SKU-005
11880,11881,ORD-011881,CUST-00382,customer11881@example.com,Europe,Australia,SKU-002,Bluetooth Speaker,Offline,M,...,0.75,0.060,16.99,Credit Card,<NA>,delivered,<NA>,ORD-011881,CUST-00382,SKU-002


In [36]:
df_messy_sales_orders.loc[
    df_messy_sales_orders["discount_pct"]== '75%']

,row_id,order_id,customer_id,customer_email,region,country,product_sku,product_name,sales_channel,order_priority,...,discount_pct,tax_rate,shipping_cost,payment_method,coupon_code,fulfillment_status,notes,order_id_key,customer_id_key,product_sku_key


In [37]:
df_messy_sales_orders.loc[3]

# .loc[conditions]
df_messy_sales_orders.loc[df_messy_sales_orders['row_id'].astype('int') < 3]

# select * from table where condition


df_messy_sales_orders.loc[df_messy_sales_orders['discount_pct'].astype('float')<0.01]

,row_id,order_id,customer_id,customer_email,region,country,product_sku,product_name,sales_channel,order_priority,...,discount_pct,tax_rate,shipping_cost,payment_method,coupon_code,fulfillment_status,notes,order_id_key,customer_id_key,product_sku_key
29,30,ORD-000030,CUST-00031,customer30@example.com,North America,UAE,SKU-007,Camping Lamp,Online,L,...,0.00,0.080,27.49,Cash,<NA>,delivered,<NA>,ORD-000030,CUST-00031,SKU-007
59,60,ORD-000060,CUST-00061,customer60@example.com,North America,Mexico,SKU-005,Skin Care Kit,Online,H,...,0.00,0.110,23.74,Credit Card,<NA>,delivered,<NA>,ORD-000060,CUST-00061,SKU-005
89,90,ORD-000090,CUST-00091,customer90@example.com,North America,Australia,SKU-003,Standing Desk,Online,L,...,0.00,0.050,19.99,Credit Card,<NA>,delivered,<NA>,ORD-000090,CUST-00091,SKU-003
102,103,ORD-000103,CUST-00104,customer103@example.com,Europe,Australia,SKU-008,SQL Handbook,Offline,C,...,-0.05,0.090,29.74,Credit Card,<NA>,delivered,<NA>,ORD-000103,CUST-00104,SKU-008
119,120,ORD-000120,CUST-00121,customer120@example.com,North America,Brazil,SKU-001,Notebook Pack,Online,H,...,0.00,0.080,16.24,wire,<NA>,delivered,<NA>,ORD-000120,CUST-00121,SKU-001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11909,11910,ORD-011910,CUST-00411,customer11910@example.com,North America,Japan,SKU-007,Camping Lamp,Online,L,...,0.00,0.080,12.49,PayPal,<NA>,delivered,<NA>,ORD-011910,CUST-00411,SKU-007
11939,11940,ORD-011940,CUST-00441,customer11940@example.com,North America,France,SKU-005,Skin Care Kit,Online,<NA>,...,0.00,0.110,FREE,Credit Card,<NA>,delivered,<NA>,ORD-011940,CUST-00441,SKU-005
11947,11948,ORD-011948,CUST-00449,customer11948@example.com,APAC,Germany,SKU-005,Skin Care Kit,Store,H,...,-0.05,0.100,14.74,CARD,<NA>,pending,<NA>,ORD-011948,CUST-00449,SKU-005
11969,11970,ORD-011970,CUST-00471,CUSTOMER11970@Example.COM,north america,Spain,SKU-003,Standing Desk,WEB,L,...,0.00,0.050,4.99,Credit Card,<NA>,CANCELLED,<NA>,ORD-011970,CUST-00471,SKU-003


APAC = Asia Pasific

In [38]:
df_messy_sales_orders_distinct_region = df_messy_sales_orders['region'].drop_duplicates()
df_messy_sales_orders_distinct_region

0             Europe
1       Asia Pacific
2      Latin America
3        Middle East
4             Africa
5      North America
18     north america
22            EUROPE
28              APAC
30             latam
36       Middle-East
40            AFRICA
210             <NA>
Name: region, dtype: string

In [ ]:
# df_messy_sales_orders['region'] = df_messy_sales_orders["region"].replace({
#         'APAC': 'Asia Pacific',
#         'latam': 'Latin America',
#         'AFRICA': 'Africa',
#         'Middle-East': 'Middle East',
#         'north america': 'North America',
#         'EUROPE': 'Europe'
#     })
# df_messy_sales_orders_distinct_region=df_messy_sales_orders['region'].drop_duplicates()
# df_messy_sales_orders_distinct_region

# df_messy_sales_orders.loc[df_messy_sales_orders['region'].astype('str')=='APAC']

,row_id,order_id,customer_id,customer_email,region,country,product_sku,product_name,sales_channel,order_priority,...,discount_pct,tax_rate,shipping_cost,payment_method,coupon_code,fulfillment_status,notes,order_id_key,customer_id_key,product_sku_key


In [39]:
df_messy_sales_orders.head()

,row_id,order_id,customer_id,customer_email,region,country,product_sku,product_name,sales_channel,order_priority,...,discount_pct,tax_rate,shipping_cost,payment_method,coupon_code,fulfillment_status,notes,order_id_key,customer_id_key,product_sku_key
0,1,ORD-000001,CUST-00002,customer1@example.com,Europe,Germany,SKU-002,Bluetooth Speaker,Offline,M,...,0.01,0.060,5.74,CARD,<NA>,delivered,<NA>,ORD-000001,CUST-00002,SKU-002
1,2,ORD-000002,CUST-00003,customer2@example.com,Asia Pacific,Japan,SKU-003,Standing Desk,Online,L,...,0.02,0.070,6.49,PayPal,<NA>,delivered,<NA>,ORD-000002,CUST-00003,SKU-003
2,3,ORD-000003,CUST-00004,customer3@example.com,Latin America,Brazil,SKU-004,Coffee Grinder,Offline,C,...,0.03,0.080,7.24,wire,<NA>,delivered,<NA>,ORD-000003,CUST-00004,SKU-004
3,4,ORD-000004,CUST-00005,customer4@example.com,Middle East,UAE,SKU-005,Skin Care Kit,Online,H,...,0.04,0.090,7.99,Cash,<NA>,delivered,<NA>,ORD-000004,CUST-00005,SKU-005
4,5,ORD-000005,CUST-00006,customer5@example.com,Africa,South Africa,SKU-006,Yoga Mat,Offline,M,...,0.05,0.100,8.74,<NA>,<NA>,delivered,<NA>,ORD-000005,CUST-00006,SKU-006


In [40]:
region_values = {
        'APAC': 'Asia Pacific',
        'latam': 'Latin America',
        'AFRICA': 'Africa',
        'Middle-East': 'Middle East',
        'north america': 'North America',
        'EUROPE': 'Europe'
    }

# row_id, order_id, and 23 columns.
# 
for value in region_values:
    print(value)
    print(region_values[value])
    print('-'*5)
    df_messy_sales_orders['region'] = df_messy_sales_orders['region'].replace(value, region_values[value])

df_messy_sales_orders_distinct_region = df_messy_sales_orders['region'].drop_duplicates()
df_messy_sales_orders_distinct_region

APAC
Asia Pacific
-----
latam
Latin America
-----
AFRICA
Africa
-----
Middle-East
Middle East
-----
north america
North America
-----
EUROPE
Europe
-----


0             Europe
1       Asia Pacific
2      Latin America
3        Middle East
4             Africa
5      North America
210             <NA>
Name: region, dtype: string

In [43]:
df_messy_sales_orders['region'] = df_messy_sales_orders["region"].replace({
        'APAC': 'Asia Pacific',
        'latam': 'Latin America',
        'AFRICA': 'Africa',
        'Middle-East': 'Middle East',
        'north america': 'North America',
        'EUROPE': 'Europe'
    })

df_messy_sales_orders['region'] = df_messy_sales_orders['region'].fillna('Unknown')
df_messy_sales_orders['region'].drop_duplicates()

0             Europe
1       Asia Pacific
2      Latin America
3        Middle East
4             Africa
5      North America
210          Unknown
Name: region, dtype: string

In [47]:
df_messy_sales_orders['sales_channel'].drop_duplicates()


sales_channel_values = {
    'online': 'Online',
    'retail': 'Retail',
    'WEB': 'Web'
}

df_messy_sales_orders['sales_channel'] = df_messy_sales_orders['sales_channel'].replace(
    {
    'online': 'Online',
    'retail': 'Retail',
    'WEB': 'Web'
    }
)

df_messy_sales_orders['sales_channel'] = df_messy_sales_orders['sales_channel'].fillna('Unknown')
df_messy_sales_orders['sales_channel'].drop_duplicates()


0      Offline
1       Online
18         Web
22      Retail
28       Store
166    Unknown
Name: sales_channel, dtype: string

In [62]:
columns = df_messy_sales_orders.columns
df_column_values = []
for column in columns:
    if column not in ('row_id', 'order_id', 'customer_id','order_id_key', 'customer_id_key', 'product_sku_key','customer_email','notes'):
        df_column_values.append(df_messy_sales_orders[column].drop_duplicates())
# for column in df_messy_sales_orders.columns:
#     for values in column:  
#         print(column)
#         print(values)
for val in df_column_values:
    print(val)

0             Europe
1       Asia Pacific
2      Latin America
3        Middle East
4             Africa
5      North America
210          Unknown
Name: region, dtype: string
0            Germany
1              Japan
2             Brazil
3                UAE
4       South Africa
5             France
6             Canada
7             Mexico
8              India
9              Spain
10             Kenya
11         Australia
12     United States
250             <NA>
Name: country, dtype: string
0      SKU-002
1      SKU-003
2      SKU-004
3      SKU-005
4      SKU-006
5      SKU-007
6      SKU-008
7      SKU-001
112       <NA>
138    sku-001
148    SKU-404
Name: product_sku, dtype: string
0      Bluetooth Speaker
1          Standing Desk
2         Coffee Grinder
3          Skin Care Kit
4               Yoga Mat
5           Camping Lamp
6           SQL Handbook
7          Notebook Pack
180                 <NA>
Name: product_name, dtype: string
0      Offline
1       Online
18         Web
